# EU-Flex

EU-Flex computes flexibility needs for a stylised three-zone European system.
The repository ships a teaser dataset (target year 2030, climate year 2013,
about 3 MB), so this notebook can plan and run without downloading anything.


In [ ]:
from pathlib import Path
import subprocess

import euflex
import pypsa

REPO = Path("/work/models/eu-flex")
print("checkout    :", (REPO / ".hem-ref").read_text().strip())
print("pypsa       :", pypsa.__version__)
print("euflex from :", Path(euflex.__file__).parent)
print("teaser data :", (REPO / "data/teaser/ty_2030_cy_2013").is_dir())


## Plan the teaser run

The workflow uses Snakemake 9 and a HiGHS solver. Planning first is a cheap
way to confirm the rules and the environment agree.


In [ ]:
result = subprocess.run(
    ["snakemake", "-n", "-c1", "--configfile", "config/teaser.yaml",
     "--directory", "runs/teaser"],
    cwd=REPO,
    capture_output=True,
    text=True,
    timeout=600,
)
print(result.stdout[-4000:])
print(result.stderr[-2000:])
assert result.returncode == 0, "the workflow could not be planned"


## Run it

```bash
run-eu-flex.sh
```

The teaser solves a full year of hourly snapshots twice (day-ahead and
intraday) across three bidding zones, so it is a real solve rather than a
smoke test. Outputs go to
`/work/models/eu-flex/runs/teaser/results/`.

## Analyse a finished run

The upstream repository ships `notebooks/teaser_case_analysis.ipynb`, which
reproduces Figures 9–18 of the ACER paper from `runs/teaser`. Open it with the
`eu-flex` kernel after the run completes.


In [ ]:
run_dir = REPO / "runs/teaser/results"
print("run finished:", run_dir.is_dir())
if run_dir.is_dir():
    for path in sorted(run_dir.rglob("*.nc"))[:10]:
        print(" ", path.relative_to(REPO))
